<a href="https://colab.research.google.com/github/pgmanuel/CAM_C04_NICE_Project/blob/pjm-dev/Intro_CAM_NICE_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Employer Project: National Institute for Health and Care Excellence (NICE)

For the Employer Project with the NICELinks to an external site., each group has been asked to build an AI-assisted method to recommend and justify clinical code lists as part of the pre-analysis phase for a given research question. This will enable NICE to better define the initial dataset more quickly and with higher confidence. They would like groups to use only publicly available materials.

The challenge is not to answer the question of whether or not an LLM can list codes. It’s to decipher whether a system can be designed to produce defensible, explainable, and reliable clinical code recommendations.’ ~ Shaun Rowark, Associate Director – Healthcare Data Analytics, NICE

Who is NICE?
The National Institute for Health and Care Excellence (NICE) exists to improve outcomes for people using the NHS and other public health and social care services by providing evidence-based guidance and advice. Its mission is to reduce variation in the quality of care across England and ensure that patients have access to effective, safe, and cost-efficient treatments.

NICE plays a central role in the English healthcare system by developing national guidance on the use of medicines, medical technologies, diagnostics, and clinical practices. One of its primary objectives is to assess the clinical and cost effectiveness of treatments to determine their suitability for use within the NHS. By evaluating both health outcomes and value for money, NICE supports the sustainable use of limited healthcare resources while maintaining high standards of care.

Internally, NICE undertakes a range of responsibilities to fulfil its role. These include commissioning and reviewing clinical evidence, conducting health economic analyses, consulting with healthcare professionals, patients, and industry stakeholders, and producing clear guidance for commissioners and practitioners. NICE also develops quality standards and performance indicators to support continuous improvement across health and social care services.

To fulfil these responsibilities, NICE maintains an internal healthcare data analytics function. This includes analysing healthcare data to develop real-world evidence, which will then be used to make decisions. In order to perform this analysis, different clinical terms need to be defined in the form of the relevant clinical codes.

The influence of NICE extends beyond England. NICE is internationally recognised as a leading authority in health technology assessment and evidence-based decision-making, and in the use of real-world evidence. Many countries and global health organisations refer to NICE’s methodologies and guidance as a benchmark for balancing innovation, affordability, and patient benefit within publicly funded healthcare systems.

To learn more about NICE and what it does, explore its website:

What NICE doesLinks to an external site.

[What NICE does](https://www.nice.org.uk/what-nice-does)

Scenario
The National Institute for Health and Care Excellence (NICE) supports evidence-based decision-making within the NHS by developing national guidance on the use of medicines, treatments, and healthcare interventions.

To produce high-impact guidance, NICE relies heavily on healthcare data analysis, particularly when assessing disease prevalence, treatment eligibility, and population-level impact. A critical component of this analysis involves defining clinical code sets, such as the Systematised Nomenclature of Medicine (SNOMED) or other diagnostic codes, which are used to identify patient cohorts within healthcare datasets.

To achieve this, NICE draws on a range of data sources, many of which are publicly available but complex to work with. These include NHS England reference sets, Quality and Outcomes Framework (QOF) – which specify clinical codes that should be used to adhere to a QOF indicator – and open clinical code repositories. The process of defining accurate and comprehensive clinical code lists for research questions is particularly challenging for two key reasons:


1.   The data sources are largely unstructured or semi-structured, often presented in formats such as spreadsheets, documents, or text-based repositories.
2.   The task requires significant domain expertise, as clinical conditions – especially those involving co-morbidities – can require hundreds of codes that must be carefully validated and justified, which can be subjective.

This project aims to improve the efficiency and accuracy of clinical code definition to support the pre-analysis phase of NICE workflows by applying advanced data science techniques, such as natural language processing (NLP) and large language models (LLMs), to these complex data sources. By enhancing NICE’s ability to generate and validate clinical code sets more effectively, the project aims to reduce manual effort, accelerate evidence generation, and ultimately improve the quality and consistency of NICE’s guidance and healthcare decision-making.

#Import data

##Import All Products

In [4]:
!pip install pinecone --quiet

In [20]:
!pip install pandas pinecone-client --quiet

In [5]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import pinecone

import warnings
warnings.filterwarnings('ignore')

In [6]:
products_data = pd.read_csv('/content/DAAR_2025_004_all_prod_codes.txt', sep='\t')


In [11]:
# Display basic info
print(f"Dataset Shape: {products_data.shape}")
products_data.head()

Dataset Shape: (1369, 2)


,code,term
0,328141000033117,Colestipol Hydrochloride Sugar free granules ...
1,565741000033115,Fenofibrate Capsules 100 mg
2,359541000033118,Colestipol Hydrochloride Sachets (orange) 5 ...
3,2994541000033117,Nicotinic Acid Titration Starter Pack M/R tab...
4,974641000033112,Nicotinic Acid Tablets 100 mg


In [12]:
# 1. Extract Manufacturer (text inside parentheses at the end of the term)
products_data['manufacturer'] = products_data['term'].str.extract(r'\((.*?)\)')

In [13]:
# 2. Identify Formulation Types (Tablet, Capsule, Solution, etc.)
def identify_formulation(term):
    term = term.lower()
    if 'tablet' in term: return 'Tablet'
    if 'capsule' in term: return 'Capsule'
    if 'suspension' in term or 'solution' in term: return 'Oral Liquid'
    if 'sachet' in term or 'powder' in term: return 'Powder/Sachet'
    if 'injection' in term: return 'Injection'
    return 'Other'

products_data['formulation_category'] = products_data['term'].apply(identify_formulation)

# 3. Identify special properties
products_data['is_sugar_free'] = products_data['term'].str.contains('sugar free', case=False)
products_data['is_modified_release'] = products_data['term'].str.contains('modified-release|XL|SR|MR|LA', case=False, regex=True)

print(f"Dataset Shape: {products_data.shape}")
products_data.head()

Dataset Shape: (1369, 6)


,code,term,manufacturer,formulation_category,is_sugar_free,is_modified_release
0,328141000033117,Colestipol Hydrochloride Sugar free granules ...,NaN,Powder/Sachet,True,False
1,565741000033115,Fenofibrate Capsules 100 mg,NaN,Capsule,False,False
2,359541000033118,Colestipol Hydrochloride Sachets (orange) 5 ...,orange,Powder/Sachet,False,False
3,2994541000033117,Nicotinic Acid Titration Starter Pack M/R tab...,NaN,Tablet,False,False
4,974641000033112,Nicotinic Acid Tablets 100 mg,NaN,Tablet,False,False


In [14]:
len(products_data)

1369

In [17]:
prod_df = products_data.drop_duplicates(subset=['term'])

In [18]:
prod_df.shape

(1363, 6)

In [19]:
products_data.to_csv('products_data.csv',index=False)

#Pinecone vector database connection

In [27]:
# Import the Pinecone library
from pinecone import Pinecone, ServerlessSpec

# Initialize the Pinecone client
pc = Pinecone(api_key="pcsk_4ZbMjE_QZHYm2hAMpnoEhvAPFfgFH5Gp61SbC1hPx7JZwZEU9K8a9xzCCEmjEzaKWkW7HV")
index = pc.Index("nice")
print("Pinecone initialized successfully!")

Pinecone initialized successfully!


In [28]:
# 2. Load the dataset
df = pd.read_csv('products_data.csv')

# Drop any rows with missing essential data (like 'code' or 'term')
df = df.dropna(subset=['code', 'term'])

print(f"Total valid records loaded: {len(df)}")

# Display the first 3 rows to verify the columns
df.head(3)

Total valid records loaded: 1369


,code,term,manufacturer,formulation_category,is_sugar_free,is_modified_release
0,328141000033117,Colestipol Hydrochloride Sugar free granules ...,NaN,Powder/Sachet,True,False
1,565741000033115,Fenofibrate Capsules 100 mg,NaN,Capsule,False,False
2,359541000033118,Colestipol Hydrochloride Sachets (orange) 5 ...,orange,Powder/Sachet,False,False


In [29]:
# 3. Define the 6 users (Namespaces)
users = [
    "student-vitor", "student-Omid", "student-Mei",
    "student-joe", "student-pedro", "student-maduraa"
]

# Calculate how many records each user gets
chunk_size = len(df) // len(users)

print(f"Distributing roughly {chunk_size} records to each of the {len(users)} users.")

Distributing roughly 228 records to each of the 6 users.


In [30]:
# 4. Process and Upsert for each user
dimension = 384 # IMPORTANT: Change this to match your Pinecone index dimension (e.g., 384, 768, 1536)

for i, user in enumerate(users):
    # Get the specific slice of data for this user
    start_idx = i * chunk_size
    # Give the last user any leftover rows
    end_idx = len(df) if i == len(users) - 1 else (i + 1) * chunk_size

    user_data = df.iloc[start_idx:end_idx]
    vectors_to_upsert = []

    # Format the data for Pinecone
    for _, row in user_data.iterrows():
        vector_id = str(row['code'])
        dummy_vector = [0.1] * dimension # Replace with actual embeddings in a real project

        metadata = {
            "term": str(row['term']),
            "manufacturer": str(row['manufacturer']) if pd.notna(row['manufacturer']) else "Unknown",
            "formulation_category": str(row['formulation_category']),
            "is_sugar_free": bool(row['is_sugar_free']),
            "is_modified_release": bool(row['is_modified_release'])
        }

        vectors_to_upsert.append((vector_id, dummy_vector, metadata))

    # Upsert in batches of 100
    batch_size = 100
    for j in range(0, len(vectors_to_upsert), batch_size):
        batch = vectors_to_upsert[j:j+batch_size]
        # Uploading to the specific user's namespace
        index.upsert(vectors=batch, namespace=user)

    print(f"✅ Upserted {len(vectors_to_upsert)} records to namespace: '{user}'")

print("\nAll data successfully uploaded!")

✅ Upserted 228 records to namespace: 'student-vitor'
✅ Upserted 228 records to namespace: 'student-Omid'
✅ Upserted 228 records to namespace: 'student-Mei'
✅ Upserted 228 records to namespace: 'student-joe'
✅ Upserted 228 records to namespace: 'student-pedro'
✅ Upserted 229 records to namespace: 'student-maduraa'

All data successfully uploaded!


In [32]:
# 5. Query the database to verify
print("Testing query for 'student-pedro'...\n")

search_results = index.query(
    vector=[0.1] * dimension,  # A dummy search vector
    top_k=3,                   # Bring back the top 3 results
    namespace="student-pedro", # ONLY search Alice's data
    include_metadata=True      # Return the product details
)

for match in search_results['matches']:
    product_name = match['metadata']['term']
    formulation = match['metadata']['formulation_category']
    print(f"Match Found: {product_name} | Type: {formulation} | Score: {match['score']:.4f}")

Testing query for 'student-pedro'...

Match Found: Hydrosaluric 50mg tablets (Organon Pharma (UK) Ltd) | Type: Tablet | Score: 1.0000
Match Found: Clonidine 100micrograms/24hours transdermal patches | Type: Other | Score: 1.0000
Match Found: Kenzem SR 120mg capsules (Kent Pharma (UK) Ltd) | Type: Capsule | Score: 1.0000


List of all indexes

In [22]:
pc.list_indexes()

[
    {
        "name": "nice",
        "metric": "cosine",
        "host": "nice-r4lsjsa.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    }
]

In [23]:
index = pc.Index('nice')

In [24]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '181',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 22 Mar 2026 16:54:05 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '88',
                                    'x-pinecone-request-latency-ms': '87',
                                    'x-pinecone-response-duration-ms': '90'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 1}},
 'storageFullness': 0.0,
 'total_vector_count': 1,
 'vector_type': 'dense'}

NotFoundException: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-10', 'x-cloud-trace-context': '67212b0b93ee7ed4c265ad3d15d46102', 'date': 'Sun, 22 Mar 2026 16:54:08 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"NOT_FOUND","message":"Resource nice-index not found"},"status":404}
